![image.png](https://i.imgur.com/a3uAqnb.png)

# Lab 2: Pruning and Distillation, Then Ship One

Lab 1 kept the network and used fewer bits per number. This lab removes numbers instead, and trains a smaller network to imitate the big one. Then you play release manager: you put every variant through a release gate and ship one. In this lab you will:

1. Train the **teacher**, Day 2's Fashion-MNIST CNN again, in about a minute.
2. **Prune** 50 to 99% of its weights, see where the cuts land, and recover the accuracy with a short fine-tune (TASK 1).
3. **Distil** it into a **student** 4x smaller, using the labels of only 1,000 images (TASK 2).
4. Measure every variant per class, on size and on speed, and write the **release gate** that picks one (TASK 3).
5. Record every variant as an **MLflow** run, and register the release in MLflow's **model registry** (TASK 4).

It runs on a laptop CPU or on Colab; no GPU needed.
Unfinished tasks raise `NotImplementedError`. The worked solution is `Optional/02_Pruning_and_Distillation_Solution.ipynb`.

## Setup

On Colab, PyTorch is already installed; the next cell adds ONNX Runtime and MLflow, used in Part 4. On your own laptop, install the course's `requirements.txt` instead and skip that cell.

In [ ]:
# Colab only. On your laptop: pip install -r requirements.txt from the course repository.
%pip install -q onnx onnxruntime "mlflow-skinny~=3.16" sqlalchemy alembic

In [ ]:
import copy
import gzip
import io
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")
torch.manual_seed(0)
print("PyTorch", torch.__version__)

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
train_set = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST("data", train=False, download=True, transform=transform)
CLASSES = train_set.classes
X_train = torch.stack([image for image, _ in train_set])    # all images in memory: 60,000 x 1 x 28 x 28
y_train = train_set.targets
X_test = torch.stack([image for image, _ in test_set])
y_test = test_set.targets
print(f"{len(X_train):,} training images, {len(X_test):,} test images, {len(CLASSES)} classes")

## Part 1: The Teacher

The same CNN as in Lab 1, trained for three epochs. A small helper trains any model on any loss, so Parts 2 and 3 can reuse it.

In [ ]:
class SimpleFashionCNN(nn.Module):
    # Day 2's Fashion-MNIST CNN: the teacher
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def train(model, epochs, loss_fn, n_images=None, lr=1e-3, seed=0):
    # mini-batch Adam on the first n_images training images (all of them by default);
    # loss_fn(model, x, y, idx) also gets the batch's positions in the training set
    generator = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    X, Y = X_train[:n_images], y_train[:n_images]
    for _ in range(epochs):
        model.train()
        for idx in torch.randperm(len(X), generator=generator).split(128):
            optimizer.zero_grad()
            loss_fn(model, X[idx], Y[idx], idx).backward()
            optimizer.step()
    return model


def cross_entropy(model, x, y, idx):
    return F.cross_entropy(model(x), y)


@torch.no_grad()
def accuracy(model):
    model.eval()
    return (model(X_test).argmax(1) == y_test).float().mean().item()


def n_params(model):
    return sum(p.numel() for p in model.parameters())


torch.manual_seed(0)
start = time.time()
teacher = train(SimpleFashionCNN(), epochs=3, loss_fn=cross_entropy)
print(f"teacher: {accuracy(teacher):.2%} test accuracy, {n_params(teacher):,} parameters, "
      f"trained in {time.time() - start:.0f} s")

## Part 2: Pruning

**Magnitude pruning** sets the smallest weights to zero, on the bet that they contribute least. **Global** pruning ranks the weights of all layers together, so each layer loses as many as it can afford, rather than the same share everywhere. PyTorch's `torch.nn.utils.prune` does it with a **mask**: the weight is multiplied by a tensor of 0s and 1s, so a pruned weight stays at zero even while training continues.

In [ ]:
def prune_global(model, amount):
    # a copy of the model with the smallest `amount` (a fraction) of all conv and linear weights set to 0
    pruned = copy.deepcopy(model)
    parameters = [(m, "weight") for m in pruned.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
    # TASK 1: zero the `amount` smallest weights by absolute value, ranked across all these layers together.
    # HINT: prune.global_unstructured(parameters, pruning_method=prune.L1Unstructured, amount=amount)
    raise NotImplementedError("TASK 1: global magnitude pruning")
    return pruned


amounts = [0.5, 0.8, 0.9, 0.95, 0.98, 0.99]
pruned_acc = [accuracy(prune_global(teacher, a)) for a in amounts]
for a, acc in zip(amounts, pruned_acc):
    print(f"{a:4.0%} of weights removed: {acc:.2%}")

plt.figure(figsize=(6, 3.5))
plt.plot([0] + amounts, [accuracy(teacher)] + pruned_acc, "o-")
plt.xlabel("fraction of weights set to zero")
plt.ylabel("test accuracy, no retraining")
plt.grid(alpha=0.3)
plt.show()

Half the weights go for free, and even 80 to 90% costs little. Past that the network breaks. Where did the cuts land?

In [ ]:
pruned95 = prune_global(teacher, 0.95)
for name, module in pruned95.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        zeros = (module.weight == 0).float().mean().item()
        print(f"{name:13s} {module.weight.numel():>8,} weights, {zeros:6.1%} pruned")

Global pruning took almost everything from the big fully connected layer, which holds 95% of the weights and was the most redundant, and only a sixth of the first convolution's 288 weights, which each do much more work. Now **fine-tune**: one more epoch, with the mask keeping the pruned weights at zero.

In [ ]:
start = time.time()
train(pruned95, epochs=1, loss_fn=cross_entropy, lr=5e-4)
print(f"95% pruned: {accuracy(prune_global(teacher, 0.95)):.2%} before fine-tuning, {accuracy(pruned95):.2%} after "
      f"({time.time() - start:.0f} s); the teacher had {accuracy(teacher):.2%}")


def bake(model):
    # make the pruning permanent: drop the masks, keep the zeros in the weights
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and prune.is_pruned(module):
            prune.remove(module, "weight")
    return model


def file_mb(model, compress=False):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    data = gzip.compress(buffer.getvalue()) if compress else buffer.getvalue()
    return len(data) / 1e6


bake(pruned95)
print(f"saved file: teacher {file_mb(teacher):.2f} MB, pruned {file_mb(pruned95):.2f} MB")
print(f"gzipped:    teacher {file_mb(teacher, True):.2f} MB, pruned {file_mb(pruned95, True):.2f} MB")

The fine-tune recovers almost all the accuracy with only 5% of the weights. But look at the saved file: **exactly the same size**. A pruned weight is still stored as a float32 zero, and the computation multiplies by it all the same. Pruning only pays off when something exploits the zeros:

- **compressed storage**: gzip above shows the potential;
- **sparse kernels**, which skip the zeros;
- **structured pruning**, which removes whole filters or neurons so the layers really shrink. That is the lecture's "structured pruning plus a fine-tune".

## Part 3: Distillation

Labels are the expensive part of a dataset. To simulate a common situation, a strong model you can query but few labels of your own, the student may use the labels of only **1,000** of the 60,000 training images. For the other 59,000 it sees the image and the teacher's prediction, nothing else.

The student is a CNN with half the channels and a quarter of the hidden units: **about 4x fewer parameters**. First, the baseline: the student trained alone on its 1,000 labelled images.

In [ ]:
class SmallCNN(nn.Module):
    # the student: half the channels, a quarter of the hidden units
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


LABELLED = 1000            # the student may use the labels of the first 1,000 training images only
print(f"student: {n_params(SmallCNN()):,} parameters ({n_params(teacher) / n_params(SmallCNN()):.1f}x fewer)")
torch.manual_seed(0)
student_alone = train(SmallCNN(), epochs=30, loss_fn=cross_entropy, n_images=LABELLED)
print(f"student alone, 1,000 labels: {accuracy(student_alone):.2%}")

Now distillation. The teacher's predictions carry more than the right answer: how sure it is, and which wrong classes look similar (a shirt resembles a T-shirt far more than a sneaker). A **temperature** $T > 1$ softens both softmaxes so those small probabilities count. The lecture's loss mixes the label term with the teacher term:

$$L = \lambda\, \mathrm{CE}\big(y,\; q_S\big) \;+\; (1-\lambda)\, T^2\, \mathrm{CE}\big(p_T^{(T)},\; q_S^{(T)}\big)$$

Here $p_T^{(T)}$ and $q_S^{(T)}$ are the teacher's and the student's softmaxes at temperature $T$. The $T^2$ undoes the $1/T^2$ shrinking of the soft term's gradients. In this setting the label term only applies to the 1,000 labelled images.

In [ ]:
with torch.no_grad():
    teacher.eval()
    teacher_logits = torch.cat([teacher(X_train[i:i + 2000]) for i in range(0, len(X_train), 2000)])


def distillation_loss(student_logits, teacher_logits, labels, has_label, T=4.0, lam=0.1):
    # TASK 2: the soft term: the student's log-probabilities at temperature T against the teacher's probabilities
    #         at temperature T, times T^2. F.kl_div gives the same gradients as the slide's cross-entropy: the two
    #         differ by the teacher's entropy, a constant for the student.
    # HINT: F.kl_div(F.log_softmax(student_logits / T, dim=1), F.softmax(teacher_logits / T, dim=1),
    #                reduction="batchmean") * T * T
    soft = None
    if soft is None:
        raise NotImplementedError("TASK 2: the distillation term")
    hard = F.cross_entropy(student_logits[has_label], labels[has_label]) if has_label.any() else 0.0
    return lam * hard + (1 - lam) * soft


def distil(model, x, y, idx):
    return distillation_loss(model(x), teacher_logits[idx], y, has_label=idx < LABELLED)


torch.manual_seed(0)
start = time.time()
student_distilled = train(SmallCNN(), epochs=2, loss_fn=distil)
print(f"student distilled from the teacher on all 60,000 images (1,000 labels): {accuracy(student_distilled):.2%} "
      f"({time.time() - start:.0f} s); trained alone it had {accuracy(student_alone):.2%}, the teacher has "
      f"{accuracy(teacher):.2%}")

The teacher turned 59,000 unlabelled images into training signal, and the student gained several points. An honest footnote: with **all** 60,000 labels, the small student trained alone does about as well as a distilled one on this dataset; we measured both within 0.3 points. Fashion-MNIST is easy enough that the labels already say what the teacher would. Distillation pays off when labels are scarce, the task is hard, or the teacher is much bigger than the student, as with DistilBERT in the lecture.

## Part 4: Ship One

You now have six candidates. Before anything ships, the lecture's release gate says to treat each one as a new model version:

- **re-run the evaluation per slice**, here per class, not just overall;
- **benchmark on the target hardware**: one image at a time, p95 latency;
- compare **size** against the device's budget.

To measure what would really ship, every variant goes through ONNX, and the int8 ones through ONNX Runtime's quantiser as in Lab 1.

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
from onnxruntime.quantization.shape_inference import quant_pre_process


def export(model, path):
    model.eval()
    torch.onnx.export(model, X_test[:1], path, input_names=["input"], output_names=["logits"],
                      dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)
    return path


class CalibrationImages(CalibrationDataReader):
    def __init__(self, n=200):
        self.images = iter([{"input": X_train[i:i + 1].numpy()} for i in range(n)])

    def get_next(self):
        return next(self.images, None)


def int8(path):
    out = path.replace(".onnx", "_int8.onnx")
    quant_pre_process(path, "prepped.onnx")
    quantize_static("prepped.onnx", out, CalibrationImages(), quant_format=QuantFormat.QDQ, per_channel=True,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
    return out


def measure(name, path):
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    session = ort.InferenceSession(path, options, providers=["CPUExecutionProvider"])
    X = X_test.numpy()
    preds = np.concatenate([session.run(None, {"input": X[i:i + 1000]})[0].argmax(1) for i in range(0, len(X), 1000)])
    per_class = np.array([(preds[y_test.numpy() == c] == c).mean() for c in range(len(CLASSES))])
    times = []
    for i in range(300):
        start = time.perf_counter()
        session.run(None, {"input": X[i:i + 1]})
        times.append((time.perf_counter() - start) * 1000)
    with open(path, "rb") as f:
        gz = len(gzip.compress(f.read())) / 1e6
    return {"variant": name, "accuracy": (preds == y_test.numpy()).mean(), "worst class": CLASSES[per_class.argmin()],
            "worst class accuracy": per_class.min(), "MB": os.path.getsize(path) / 1e6, "MB gzipped": gz,
            "p95 ms": np.percentile(times[20:], 95), "per class": per_class}


variants = {
    "teacher, float32": export(teacher, "teacher.onnx"),
    "teacher, int8": int8("teacher.onnx"),
    "teacher pruned 95% + fine-tune": export(pruned95, "pruned95.onnx"),
    "student alone, float32": export(student_alone, "student_alone.onnx"),
    "student distilled, float32": export(student_distilled, "student_distilled.onnx"),
    "student distilled, int8": int8("student_distilled.onnx"),
}
table = pd.DataFrame([measure(name, path) for name, path in variants.items()]).set_index("variant")
print(table.drop(columns="per class").round(3).to_string())

### The gate

A product team gives you the budget. The model must:

- lose at most **3 points** of overall accuracy against the teacher;
- lose at most **10 points** on the teacher's **worst class**. Compression tends to cost the rare and hard cases most, which is the Responsible AI lecture's learning bias, now in deployment;
- fit in **0.2 MB**, the device's storage for the model;
- answer one image in **2 ms at p95**, on this machine.

Write the check, then ship the **smallest** variant that passes.

In [ ]:
GATE = {"max_accuracy_drop": 0.03, "max_worst_class_drop": 0.10, "max_mb": 0.2, "max_p95_ms": 2.0}
teacher_row = table.loc["teacher, float32"]


def passes_gate(row):
    # TASK 3: True only if the variant meets every line of GATE. Accuracy drops are measured against the teacher:
    #         overall (teacher_row["accuracy"]), and on the teacher's worst class (teacher_row["worst class accuracy"]).
    #         Compare file size ("MB") and latency ("p95 ms") with their budgets.
    raise NotImplementedError("TASK 3: the release gate")


table["passes"] = table.apply(passes_gate, axis=1)
print(table[["accuracy", "worst class accuracy", "MB", "p95 ms", "passes"]].round(3).to_string())
shipped = table[table["passes"]].sort_values("MB")
print("\nship:", shipped.index[0] if len(shipped) else "nothing passes; renegotiate the budget or try harder")

### Record every variant in MLflow

A table printed in a notebook is gone the moment the kernel restarts. The lecture's answer is **experiment tracking**: each variant becomes an MLflow **run**, holding the parameters that made it, the metrics it scored, and the model file itself. MLflow keeps them here in a local SQLite file, `mlflow.db`: no server and no account. Every run in this session gets the same `session` tag, so re-running the notebook does not mix old runs into the table.

In [ ]:
import logging
import mlflow
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from pathlib import Path

logging.getLogger("mlflow").setLevel(logging.WARNING)
mlflow.set_tracking_uri("sqlite:///mlflow.db")                 # one local file: no server, no account
EXPERIMENT = "day5-compression"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=Path("mlartifacts").resolve().as_uri())
mlflow.set_experiment(EXPERIMENT)
SESSION = time.strftime("%Y-%m-%d %H:%M:%S")

DETAILS = {   # how each variant was made: what a colleague needs to reproduce it
    "teacher, float32": {"architecture": "SimpleFashionCNN", "method": "none", "bits": 32},
    "teacher, int8": {"architecture": "SimpleFashionCNN", "method": "post-training quantisation", "bits": 8},
    "teacher pruned 95% + fine-tune": {"architecture": "SimpleFashionCNN", "method": "global magnitude pruning",
                                       "bits": 32, "sparsity": 0.95, "fine_tune_epochs": 1},
    "student alone, float32": {"architecture": "SmallCNN", "method": "trained alone", "bits": 32,
                               "labelled_images": LABELLED},
    "student distilled, float32": {"architecture": "SmallCNN", "method": "distillation", "bits": 32,
                                   "labelled_images": LABELLED, "temperature": 4.0, "lambda": 0.1},
    "student distilled, int8": {"architecture": "SmallCNN", "method": "distillation + post-training quantisation",
                                "bits": 8, "labelled_images": LABELLED, "temperature": 4.0, "lambda": 0.1},
}

run_ids = {}
for variant, row in table.iterrows():
    with mlflow.start_run(run_name=variant) as run:
        mlflow.set_tags({"session": SESSION, "gate": "passed" if row["passes"] else "failed"})
        mlflow.log_params(DETAILS[variant])
        mlflow.log_metrics({"accuracy": row["accuracy"], "worst_class_accuracy": row["worst class accuracy"],
                            "size_mb": row["MB"], "size_mb_gzipped": row["MB gzipped"], "p95_ms": row["p95 ms"]})
        mlflow.log_metrics({"accuracy_" + name.replace("/", "_").replace(" ", "_"): acc
                            for name, acc in zip(CLASSES, row["per class"])})
        mlflow.log_artifact(variants[variant], artifact_path="model")       # the ONNX file itself
        run_ids[variant] = run.info.run_id

history = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string=f"tags.session = '{SESSION}'")
columns = {"tags.mlflow.runName": "variant", "params.method": "method", "metrics.accuracy": "accuracy",
           "metrics.worst_class_accuracy": "worst class", "metrics.size_mb": "MB", "metrics.p95_ms": "p95 ms",
           "tags.gate": "gate"}
print(history[list(columns)].rename(columns=columns).round(3).to_string(index=False))

On your own laptop, `pip install mlflow` (the full package, with its web interface) and run `mlflow ui --backend-store-uri sqlite:///mlflow.db` in this folder to browse the same runs in a browser.

### Register the release

Tracking records what you tried; the **model registry** records what you ship. Each registered **version** points at one run's model file and never changes. **Aliases** are movable labels on versions. The device serves whatever `champion` points to. `rollback` keeps the uncompressed original one command away, as the lecture's release slide asks.

In [ ]:
client = MlflowClient()
MODEL = "fashion-mnist-classifier"
try:
    client.create_registered_model(MODEL, description="Day 5 Lab 2: the classifier and its compressed variants")
except MlflowException:
    pass                                    # registered by an earlier run of this notebook


def register(variant, alias):
    run_id = run_ids[variant]
    # TASK 4: create a model version from the variant's run, then point `alias` at that version.
    # HINT: version = client.create_model_version(MODEL, source=f"runs:/{run_id}/model", run_id=run_id),
    #       then client.set_registered_model_alias(MODEL, alias, version.version)
    raise NotImplementedError("TASK 4: register the variant")
    return version.version


register("teacher, float32", "rollback")        # the uncompressed original: the way back
register(shipped.index[0], "champion")          # what the device serves
for alias in ("champion", "rollback"):
    version = client.get_model_version_by_alias(MODEL, alias)
    run = mlflow.get_run(version.run_id)
    print(f"{alias:9s} -> version {version.version}: {run.data.tags['mlflow.runName']}, "
          f"accuracy {run.data.metrics['accuracy']:.2%}, {run.data.metrics['size_mb']:.3f} MB")

In [ ]:
plt.figure(figsize=(10, 3.5))
width = 0.4
x = np.arange(len(CLASSES))
plt.bar(x - width / 2, table.loc["teacher, float32", "per class"], width, label="teacher, float32")
plt.bar(x + width / 2, table.loc["student distilled, int8", "per class"], width, label="student distilled, int8")
plt.xticks(x, CLASSES, rotation=30)
plt.ylim(0.4, 1.0)
plt.ylabel("accuracy per class")
plt.legend()
plt.tight_layout()
plt.show()

**Questions to discuss:**

1. Which line of the gate does each rejected variant fail?
2. The pruned teacher's file is too big, but gzipped it would fit. What would you need on the device to ship it?
3. Compare the per-class bars. Did the student inherit the teacher's weak classes? The lecture warns that it does.
4. The lecture's order of effort is: quantise first (minutes, no retraining), then prune and fine-tune, then distil. With a 0.5 MB budget instead of 0.2 MB, where would you have stopped?
5. A week after shipping, the device starts misclassifying shirts. Using the registry, what do you change to roll back, and what never changes?

## Takeaways

- **Pruning:** most weights of an over-parameterised layer can go; global ranking finds them; a short fine-tune recovers the accuracy.
- **Unstructured zeros save nothing until something exploits them:** compressed storage, sparse kernels, or structured pruning.
- **Distillation transfers what the teacher knows**, including to images nobody labelled. It pays off most when labels are scarce or the teacher is far bigger than the student.
- **Compressed models are new model versions.** Gate them per class, on size and on the target hardware, and ship the smallest one that passes.
- **Track every variant, register what ships.** Runs keep the evidence; registry aliases decide what serves, and what you roll back to.